In [13]:
#ON PEUT NE AS L'EXECUTER NEXT TIME 
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
from pyspark.sql.window import Window
# ============================================
# ÉTAPE 0 : LECTURE ET EXPLORATION DES DONNÉES
# ============================================
storage_account = "energybigdatastorage"
container_raw = "raw"
path_raw = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/halfhourly_dataset/halfhourly_dataset"
# Lire 10000 lignes pour explorer
df_hh = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .load(path_raw) \

print(f"Nombre de lignes : {df_hh.count()}")
print(f"Nombre de colonnes : {len(df_hh.columns)}")
print("\n=== SCHEMA ===")
df_hh.printSchema()
print("\n=== APERÇU ===")
df_hh.show(5)
print("\n=== VALEURS NULLES ===")
df_hh.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_hh.columns]).show()
print("\n=== STATISTIQUES ===")
df_hh.describe().show()
print("\n=== DOUBLONS ===")
print(f"Nombre de doublons : {df_hh.count() - df_hh.dropDuplicates().count()}")

In [14]:
# ============================================
# ÉTAPE 1 : SUPPRIMER LES DOUBLONS
# ============================================
nb_avant = df_hh.count()
df_hh = df_hh.dropDuplicates()
nb_apres = df_hh.count()
print(f"Lignes avant : {nb_avant}")
print(f"Lignes après : {nb_apres}")
print(f"Doublons supprimés : {nb_avant - nb_apres}")

In [15]:
# ============================================
# ÉTAPE 2 : STANDARDISER LES UNITÉS
# ============================================

# Remplacer les "Null" string par de vrais nulls d'abord
df_hh = df_hh.withColumn("energy(kWh/hh)",
    F.when(F.col("energy(kWh/hh)") == "Null", None)
    .otherwise(F.col("energy(kWh/hh)")))

# Convertir energy(kWh/hh) de string à float
df_hh = df_hh.withColumn("energy_kwh", 
    F.col("energy(kWh/hh)").cast(FloatType()))

# Supprimer l'ancienne colonne
df_hh = df_hh.drop("energy(kWh/hh)")

print("=== SCHEMA APRÈS CONVERSION ===")
df_hh.printSchema()
df_hh.show(5)

In [16]:
# ============================================
# ÉTAPE 3 : TRAITEMENT DES VALEURS NULLES
# ============================================
print("=== VALEURS NULLES PAR COLONNE ===")
df_hh.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_hh.columns]).show()

# Forward fill sur energy_kwh car séries temporelles
window_spec = Window.partitionBy("LCLid").orderBy("tstp")
df_hh = df_hh.withColumn("energy_kwh",
    F.last("energy_kwh", ignorenulls=True).over(window_spec))

print("=== VALEURS NULLES APRÈS TRAITEMENT ===")
df_hh.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_hh.columns]).show()

In [17]:
# ============================================
# ÉTAPE 4 : CORRECTION DES FORMATS DE DATES
# ============================================
print("=== FORMAT DE DATE AVANT ===")
df_hh.select("tstp").show(5)
print(f"Type de tstp : {df_hh.schema['tstp'].dataType}")

# Vérifier si tstp est déjà en timestamp
# Si oui, pas besoin de conversion
# Si non, convertir
df_hh = df_hh.withColumn("tstp", F.to_timestamp("tstp"))

print("=== FORMAT DE DATE APRÈS ===")
df_hh.select("tstp").show(5)
print(f"Type de tstp : {df_hh.schema['tstp'].dataType}")

In [18]:
# ============================================
# ÉTAPE 5 : ANALYSE DES OUTLIERS
# ============================================
print("=== STATISTIQUES energy_kwh ===")
df_hh.select(
    F.min("energy_kwh").alias("min"),
    F.max("energy_kwh").alias("max"),
    F.mean("energy_kwh").alias("moyenne"),
    F.stddev("energy_kwh").alias("ecart_type")
).show()

# Calculer Q1, Q3 et IQR
quantiles = df_hh.approxQuantile("energy_kwh", [0.25, 0.75], 0.05)
Q1 = quantiles[0]
Q3 = quantiles[1]
IQR = Q3 - Q1

seuil_bas = Q1 - 1.5 * IQR
seuil_haut = Q3 + 1.5 * IQR

print(f"Q1 : {Q1}")
print(f"Q3 : {Q3}")
print(f"IQR : {IQR}")
print(f"Seuil bas : {seuil_bas}")
print(f"Seuil haut : {seuil_haut}")

nb_outliers = df_hh.filter(
    (F.col("energy_kwh") < seuil_bas) | 
    (F.col("energy_kwh") > seuil_haut)
).count()

print(f"\nNombre d'outliers détectés : {nb_outliers}")
print(f"Pourcentage : {round(nb_outliers/df_hh.count()*100, 2)}%")

In [19]:
# ============================================
# ÉTAPE 6 : EXTRACTION DES FEATURES TEMPORELLES
# ============================================
df_hh = df_hh.withColumn("hour", F.hour("tstp"))
df_hh = df_hh.withColumn("day_of_week", F.dayofweek("tstp"))
df_hh = df_hh.withColumn("month", F.month("tstp"))
df_hh = df_hh.withColumn("year", F.year("tstp"))

print("=== SCHEMA APRÈS EXTRACTION ===")
df_hh.printSchema()
print("\n=== APERÇU AVEC FEATURES TEMPORELLES ===")
df_hh.show(5)

In [20]:
# ============================================
# ÉTAPE 7 : CONSOMMATION MOYENNE PAR FOYER
# ============================================
df_avg = df_hh.groupBy("LCLid").agg(
    F.mean("energy_kwh").alias("avg_energy_kwh"),
    F.min("energy_kwh").alias("min_energy_kwh"),
    F.max("energy_kwh").alias("max_energy_kwh"),
    F.stddev("energy_kwh").alias("std_energy_kwh"),
    F.count("energy_kwh").alias("nb_mesures")
)

print("=== CONSOMMATION MOYENNE PAR FOYER ===")
df_avg.show(10)
print(f"\nNombre de foyers : {df_avg.count()}")

In [21]:
# ============================================
# ÉTAPE 8 : SAUVEGARDE DANS PROCESSED
# ============================================
container_processed = "processed"

# Chemin pour les données nettoyées
path_processed_hh = f"abfss://{container_processed}@{storage_account}.dfs.core.windows.net/halfhourly/"

# Chemin pour la consommation moyenne par foyer
path_processed_avg = f"abfss://{container_processed}@{storage_account}.dfs.core.windows.net/halfhourly_avg/"

# Sauvegarder en format Delta Lake
df_hh.write.format("delta").mode("overwrite").save(path_processed_hh)
print(" halfhourly nettoyé sauvegardé dans processed/halfhourly/")

df_avg.write.format("delta").mode("overwrite").save(path_processed_avg)
print(" Consommation moyenne sauvegardée dans processed/halfhourly_avg/")

In [3]:
from pyspark.sql import functions as F

# Vérification des données sauvegardées
storage_account = "energybigdatastorage"

path_processed_hh = f"abfss://processed@{storage_account}.dfs.core.windows.net/halfhourly/"
path_processed_avg = f"abfss://processed@{storage_account}.dfs.core.windows.net/halfhourly_avg/"

# Lire les données nettoyées
df_verification = spark.read.format("delta").load(path_processed_hh)
df_avg_verification = spark.read.format("delta").load(path_processed_avg)

print("=== HALFHOURLY NETTOYÉ ===")
print(f"Nombre de lignes : {df_verification.count()}")
df_verification.printSchema()
df_verification.show(5)

print("\n=== CONSOMMATION MOYENNE PAR FOYER ===")
print(f"Nombre de foyers : {df_avg_verification.count()}")
df_avg_verification.show(5)

print("\n=== VÉRIFICATION NULLS ===")
df_verification.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_verification.columns]).show()